# Solutions · Chapter 05-08 · Bias, variance, and learning curves

Worked answers to every exercise in `notebooks/05_regression/05-08_bias_variance.ipynb`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")


def true_function(x):
    return np.sin(1.2 * x) * 3 + 0.5 * x


NOISE_SD = 1.5
EVALUATION_GRID = np.linspace(-4, 4, 60)


def fits_for(degree, rows_per_dataset, datasets, seed=0):
    """Fit the same model on `datasets` fresh samples; return the predictions."""
    rng = np.random.default_rng(seed)
    predictions = np.zeros((datasets, len(EVALUATION_GRID)))
    for run in range(datasets):
        x = rng.uniform(-4, 4, rows_per_dataset)
        y = true_function(x) + rng.normal(0, NOISE_SD, rows_per_dataset)
        model = make_pipeline(PolynomialFeatures(degree, include_bias=False),
                              StandardScaler(), LinearRegression())
        predictions[run] = model.fit(x.reshape(-1, 1),
                                     y).predict(EVALUATION_GRID.reshape(-1, 1))
    return predictions


def terms_of(predictions):
    bias_squared = float(((predictions.mean(axis=0)
                           - true_function(EVALUATION_GRID)) ** 2).mean())
    return bias_squared, float(predictions.var(axis=0).mean())


print("noise floor: variance %.2f, RMSE %.2f" % (NOISE_SD ** 2, NOISE_SD))

## Quick understanding

### E1 · The decomposition

$$\mathbb{E}\left[(y - \hat{f}(x))^2\right] = \text{bias}^2 + \text{variance} + \sigma^2$$

- **bias²** - how far the *average* fit is from the truth: the model is too stiff to reach it.
- **variance** - how much the fits disagree with *each other* across training samples.
- **σ²** - the noise in the target, which no model can predict.

### E2 · Which lever moves which term

**More data reduces variance.** Every extra row constrains the coefficients further, so the fits from
different samples converge on each other. It does **nothing** to bias: a straight line fitted to a million
points from a sine wave is still a straight line.

**More capacity reduces bias** - if it is capacity of the right shape - and **increases variance**,
because more parameters means more that can be set by whichever rows you happened to get.

That is the whole trade-off, and the reason the two failures of 05-07 need opposite fixes.

### E3 · Why averaging kills variance and not bias

Variance is spread **about the average fit**. Averaging many fits is literally computing that average, and
the spread of an average of `k` independent estimates is about `1/k` of the spread of one.

Bias is the distance **of the average fit from the truth**. Averaging more fits estimates that same
average more precisely - it does not move it. Average a thousand straight lines fitted to a sine wave and
you get a straight line.

> **You can average your way out of variance. You cannot average your way out of bias.**

This is exactly the mechanism of a random forest (05-10) - many high-variance, low-bias trees averaged -
and it is why the same trick applied to many linear models would achieve nothing.

## Hand calculation

### E4 · Five predictions, truth 10

Predictions 12, 14, 11, 15, 13.

**Mean prediction** = `(12 + 14 + 11 + 15 + 13) / 5 = 65 / 5 = ` **13**

**Bias** = `13 - 10 = ` **3**   →   **bias² = 9**

**Variance**: deviations from 13 are `-1, +1, -2, +2, 0`; their squares are `1, 1, 4, 4, 0`, summing to
10, so variance = `10 / 5 = ` **2**.

Note both are computed about the *mean prediction*, not about the truth. Confusing the two is the usual
slip: the spread about the truth would be `(4 + 16 + 1 + 25 + 9)/5 = 11`, which is `9 + 2` - bias² plus
variance, the decomposition in miniature.

### E5 · Expected squared error at that point

`bias² + variance + noise = 9 + 2 + 4 = ` **15**

**Bias dominates at 9 of 15 - 60% of the error.** The model is wrong in the same direction every time,
and the fix is a better model rather than more data or an ensemble.

### E6 · A (bias² 9, variance 1) against B (bias² 1, variance 9)

**Both total `9 + 1 + 4 = 14` and `1 + 9 + 4 = 14`. On expected squared error alone there is nothing to
choose.**

**I would take B**, and the reasons are all about what happens next:

- **B improves with more data; A does not.** B's variance term is the one that shrinks as rows are added,
  so B is on a path and A is at a dead end.
- **B improves with averaging.** Fit B on twenty resamples and average, and its variance term falls
  towards 1 while its bias stays at 1 - a total near 6. The same treatment leaves A at 14.
- **A is wrong in a consistent direction**, which is often worse operationally than being wrong in
  varying directions: a systematic error compounds when predictions are summed or acted on repeatedly.

**What would change my answer:**

- **If no more data is coming and a single fit must be shipped**, the two are genuinely equal, and I would
  pick on interpretability and cost.
- **If the loss is not squared error.** A high-variance model produces occasional very large errors, and
  under an asymmetric or bounded loss those may be unacceptable even though the mean square is the same.
- **If A's bias is in a known direction**, it can be corrected with an offset - which converts A into the
  better model outright.

### E7 · Train 3.0, validation 3.1, floor 1.0

**Gap 0.1: the curves have converged. Meeting point 3.1 against a floor of 1.0: 2.1 of unexplained
error.**

**Diagnosis: underfitting, decisively.** Variance has been squeezed out - the gap is essentially zero at
1,000 rows - and what remains is bias.

**Next action: more capacity, not more data.** Concretely, in order: add interactions or non-linear terms
for the features you have (05-07); try a flexible model class - gradient boosting is the cheap test; then
go looking for features you do not have, because a 2.1 shortfall that survives a flexible model is
usually a missing column rather than a missing algorithm.

**What I would not do is collect more rows.** The curve has already told you what they would buy: nothing.

## Coding

### E8 · A reusable decomposition

In [ ]:
def bias_variance(model_factory, truth_fn, n_rows, datasets=400, noise_sd=NOISE_SD,
                  grid=EVALUATION_GRID, seed=0):
    rng = np.random.default_rng(seed)
    predictions = np.zeros((datasets, len(grid)))
    measured = []
    for run in range(datasets):
        x = rng.uniform(grid.min(), grid.max(), n_rows)
        y = truth_fn(x) + rng.normal(0, noise_sd, n_rows)
        fitted = model_factory().fit(x.reshape(-1, 1), y)
        predictions[run] = fitted.predict(grid.reshape(-1, 1))
        fresh = truth_fn(grid) + rng.normal(0, noise_sd, len(grid))
        measured.append(float(((fresh - predictions[run]) ** 2).mean()))
    bias_squared = float(((predictions.mean(axis=0) - truth_fn(grid)) ** 2).mean())
    variance = float(predictions.var(axis=0).mean())
    return {"bias squared": bias_squared, "variance": variance,
            "noise": noise_sd ** 2, "sum": bias_squared + variance + noise_sd ** 2,
            "measured": float(np.mean(measured))}


def polynomial(degree):
    return lambda: make_pipeline(PolynomialFeatures(degree, include_bias=False),
                                 StandardScaler(), LinearRegression())


checks = []
for label, factory in [("degree 2", polynomial(2)), ("degree 6", polynomial(6))]:
    row = bias_variance(factory, true_function, n_rows=40)
    row["model"] = label
    checks.append(row)
frame = pd.DataFrame(checks)[["model", "bias squared", "variance", "noise",
                              "sum", "measured"]]
frame["difference"] = frame["measured"] - frame["sum"]
print(frame.to_string(index=False, float_format=lambda v: "%.4f" % v))

**The identity holds for both** - the measured error and `bias² + variance + noise` differ by -0.0023
and +0.0218, on totals of 7.38 and 3.36.

Two details in the implementation matter and are easy to get wrong.

**The fresh noise must be drawn separately for the measurement.** `measured` compares each fit against a
*newly noised* target, not against the truth - otherwise the σ² term is absent and the identity does not
close. That line is the operational meaning of "the noise is irreducible".

**`predictions.var(axis=0)` uses the population variance**, dividing by `datasets` rather than
`datasets - 1`. The decomposition is stated in terms of the population quantity; with 400 datasets the
difference is a quarter of a percent, but on 20 it would be 5%.

### E9 · The same sweep on 200 rows

In [ ]:
sweep = []
for degree in [1, 2, 3, 5, 8, 12, 16]:
    small_bias, small_variance = terms_of(fits_for(degree, 30, 400))
    large_bias, large_variance = terms_of(fits_for(degree, 200, 400))
    sweep.append({"degree": degree,
                  "bias2 @30": small_bias, "var @30": small_variance,
                  "total @30": small_bias + small_variance + NOISE_SD ** 2,
                  "bias2 @200": large_bias, "var @200": large_variance,
                  "total @200": large_bias + large_variance + NOISE_SD ** 2})
sweep = pd.DataFrame(sweep)
print(sweep.to_string(index=False, float_format=lambda v: "%.4f" % v))

for label, column in [("30 rows", "total @30"), ("200 rows", "total @200")]:
    winner = sweep.loc[sweep[column].idxmin()]
    print("\nbest at %s: degree %d, total %.4f"
          % (label, winner["degree"], winner[column]))

**The crossover moves up, and the reason is visible column by column.**

**The bias columns barely move.** Degree 3 has bias² 1.0922 on 30 rows and 1.0411 on 200; degree 5 has
0.0458 and 0.0343. **Bias is a property of the model, not of the sample size** - a straight line is
equally unable to be a sine wave at any n.

**The variance columns collapse.** Degree 8 goes from **172.59 to 0.13**, a factor of 1,300. Degree 16
goes from 154 million to 1.30.

Since the total is bias + variance + noise, and only variance responds to `n`, the balance tips: with
plenty of rows the extra capacity is nearly free, so the best degree is whichever has the least bias you
can afford. At 30 rows degree 5 wins at 3.53; at 200 rows degree 5 and degree 8 are effectively tied at
2.37 and 2.38, and both are close to the floor of 2.25.

**This is 05-07's sample-size finding with the mechanism supplied.** There it was an observation about
held-out R-squared; here it is a statement about which of two terms is doing the damage, and why only one
of them cares how much data you have.

### E10 · A learning curve on real data

In [ ]:
california = fetch_california_housing(as_frame=True)
folds = KFold(5, shuffle=True, random_state=0)

sizes = np.array([200, 800, 3200, 8000, 15480])
counts, train_scores, validation_scores = learning_curve(
    make_pipeline(StandardScaler(), LinearRegression()),
    california.data, california.target, train_sizes=sizes, cv=folds,
    scoring="neg_root_mean_squared_error")
train_line, validation_line = -train_scores.mean(axis=1), -validation_scores.mean(axis=1)

print(pd.DataFrame({"rows used": counts, "train RMSE": train_line,
                    "validation RMSE": validation_line,
                    "gap": validation_line - train_line})
      .to_string(index=False, float_format=lambda v: "%.4f" % v))

boosted = -cross_val_score(HistGradientBoostingRegressor(random_state=0),
                           california.data, california.target, cv=folds,
                           scoring="neg_root_mean_squared_error").mean()
print("\na flexible model on the same data and folds: RMSE %.4f" % boosted)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(counts, train_line, "o-", color="#0072B2", linewidth=2.4, markersize=9,
        label="on the rows it was fitted to")
ax.plot(counts, validation_line, "s-", color="#D55E00", linewidth=2.4, markersize=9,
        label="on rows it has not seen")
ax.axhline(boosted, color="#009E73", linestyle="--", linewidth=2,
           label="gradient boosting reaches %.3f" % boosted)
ax.set_xscale("log")
ax.set_xticks(counts)
ax.set_xticklabels(["%d" % c for c in counts], fontsize=9)
ax.minorticks_off()
ax.set_ylim(0.35, 2.2)
ax.annotate("validation at 200 and 800 rows\nis 15.30 and 5.18 - off the top.\nReal curves are wild at the left end.",
            xy=(3200, 0.942), xytext=(230, 1.55), fontsize=9, color="#D55E00",
            arrowprops=dict(arrowstyle="->", color="#D55E00", linewidth=1.4))
ax.annotate("and not monotone", xy=(8000, 1.2753), xytext=(4200, 1.85),
            fontsize=9, color="#D55E00",
            arrowprops=dict(arrowstyle="->", color="#D55E00", linewidth=1.4))
ax.set_xlabel("block groups used to fit (log scale)")
ax.set_ylabel("RMSE ($100k)")
ax.set_title("California: the curves met, and a better model is well below them",
             fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**No - I would not collect more block groups, and the curve says so twice over.**

**The gap is gone.** At 15,480 rows the training and validation RMSEs are 0.7331 and 0.7269, a gap of
**-0.006**. There is no variance left to buy off. Another twenty thousand rows would move this by nothing
measurable.

**And the meeting point is not the floor.** Gradient boosting on the identical data and folds reaches
**0.4668** against the linear model's 0.7263 - **36% lower.** So at least 0.26 of the linear model's error
is bias rather than irreducible noise, and it is addressable by a better model rather than more rows.

**The honest reading of the small-`n` end.** Validation RMSE is **15.30** at 200 rows and 5.18 at 800 -
absurd values that do not appear in any textbook picture. This is real data: California Housing has
extreme rows (`AveRooms` reaches 141, `AveOccup` reaches 1243), and a linear model fitted on 200 rows
extrapolates catastrophically on them. There is a non-monotone bump at 8,000 too.

**Do not tidy that away.** The lesson is that real learning curves are noisy at the left end, and the
reading is done on the right-hand end where the curve has settled. If you need the left end to be
trustworthy, average over more folds and more repeats.

### E11 · Averaging twenty-five fits

In [ ]:
many = fits_for(8, rows_per_dataset=30, datasets=500)
single_bias, single_variance = terms_of(many)

ensembles = many.reshape(20, 25, len(EVALUATION_GRID)).mean(axis=1)
ensemble_bias, ensemble_variance = terms_of(ensembles)

print("one degree-8 model     : bias%s %.4f   variance %10.4f"
      % (chr(178), single_bias, single_variance))
print("the average of 25      : bias%s %.4f   variance %10.4f"
      % (chr(178), ensemble_bias, ensemble_variance))
print("\nvariance fell by a factor of %.1f (25 would be the ideal)"
      % (single_variance / ensemble_variance))
print("bias changed by %.6f" % (round(ensemble_bias - single_bias, 6) + 0.0))

**Variance fell from 161.90 to 7.29 - a factor of 22.2 - and bias² did not move at all: 0.1349 to
0.1349.**

The theoretical factor is 25, one over the number of fits averaged, and 22.2 is that estimate measured
from only 20 ensembles. The bias is unchanged to four decimal places, which is not an approximation but
the definition: the average of averages *is* the average.

**The practical consequence is large.** A single degree-8 model has a total error of about
`0.13 + 161.90 + 2.25 = 164.3`. The average of 25 has `0.13 + 7.29 + 2.25 = 9.7`, a seventeen-fold
improvement - **from the same model class, with no tuning, on the same data.**

**This is a random forest, in principle.** Trees have low bias and enormous variance, exactly like the
degree-8 polynomial, so averaging them is the intervention that fits the problem. 05-10 builds one.

**And the caveat that makes it honest:** the 25 fits here were on **independent samples**, which is a
luxury nobody has. Bagging approximates it by resampling one dataset with replacement, so the fits are
correlated and the reduction is less than `1/k`. The direction is right, the factor is optimistic.

### E12 · Noisier rows against shifted rows

In [ ]:
supplier_rng = np.random.default_rng(23)
CLEAN_ROWS, total_rows = 300, 2000
supply_x = supplier_rng.uniform(-4, 4, total_rows)
clean_y = true_function(supply_x) + supplier_rng.normal(0, NOISE_SD, total_rows)
from_new = np.arange(total_rows) >= CLEAN_ROWS

shifted_y = clean_y.copy()
shifted_y[from_new] = (0.7 * true_function(supply_x[from_new]) + 4.0
                       + supplier_rng.normal(0, NOISE_SD, from_new.sum()))

noisier_y = clean_y.copy()
noisier_y[from_new] = (true_function(supply_x[from_new])
                       + supplier_rng.normal(0, NOISE_SD * 4, from_new.sum()))

check_x = supplier_rng.uniform(-4, 4, 600)
check_y = true_function(check_x) + supplier_rng.normal(0, NOISE_SD, 600)

comparison = []
for used in [100, 300, 500, 800, 1200, 2000]:
    row = {"rows used": used}
    for label, target in [("shifted", shifted_y), ("noisier", noisier_y)]:
        model = make_pipeline(PolynomialFeatures(5, include_bias=False),
                              StandardScaler(), LinearRegression())
        model.fit(supply_x[:used].reshape(-1, 1), target[:used])
        row[label] = float(np.sqrt(
            ((check_y - model.predict(check_x.reshape(-1, 1))) ** 2).mean()))
    comparison.append(row)
print(pd.DataFrame(comparison).to_string(index=False,
                                         float_format=lambda v: "%.4f" % v))
print("\nthe noise floor is %.2f" % NOISE_SD)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
frame = pd.DataFrame(comparison)
ax.plot(frame["rows used"], frame["shifted"], "o-", color="#D55E00", linewidth=2.6,
        markersize=10, label="the new rows are SHIFTED")
ax.plot(frame["rows used"], frame["noisier"], "s-", color="#0072B2", linewidth=2.6,
        markersize=10, label="the new rows are merely NOISIER")
ax.axhline(NOISE_SD, color="#000000", linestyle="--", linewidth=1.8,
           label="noise floor, %.2f" % NOISE_SD)
ax.axvline(CLEAN_ROWS, color="#666666", linewidth=1.6, linestyle=":",
           label="the new supplier starts here")
ax.set_xlabel("rows used to fit")
ax.set_ylabel("validation RMSE on clean rows")
ax.set_title("Noisier labels are survivable. Shifted ones are not.", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**No - the curve does not turn upward when the new rows are merely noisier. It goes flat.**

Noisier rows: **1.5041, 1.5124, 1.5273, 1.5526, 1.5124, 1.5207** - hovering at the floor throughout, with
no trend. Shifted rows climb to 3.83.

**The difference is whether the extra rows are wrong on average or merely uncertain.**

A noisier row still has a target **centred on the truth**. Least squares averages over rows, and the
average of many unbiased-but-noisy observations is still unbiased - so the fit stays in the right place,
it just learns less per row than a clean one would. Adding them cannot make the model worse; at worst it
stops making it better.

A shifted row has a target centred somewhere **else**. Now the average is pulled, and every additional
row pulls it further. **The damage grows with the amount of data**, which is the shape that produces the
upward curve.

> **Noise costs you information. A shift costs you correctness. Only the second is a reason to throw data
> away.**

**Which is a genuinely useful thing to know when triaging a data source.** "This vendor's labels are
sloppier" is usually tolerable - dilution, not poison. "This vendor measures a slightly different thing"
is not, and the two sound similar in a meeting.

**How to tell them apart on real data:** fit on the old rows only and on the new rows only, and compare
the *predictions* of the two models on a common set. Similar predictions with different errors means
noise; systematically different predictions means a shift.

## Interpretation

### E13 · The lines are still far apart and both still falling

**What I report: the curve is still descending, so more data will help - and here is how much, and what
it costs.**

The answer is not "yes" but a number, and E20's technique gets it: fit `excess = a * n^b` to the
validation curve, extrapolate to the sizes under discussion, and put that next to the price per row.

> "At 2,000 rows we are at RMSE 4.1. The trend puts 5,000 rows at about 3.4 and 10,000 at about 3.0. At
> 8 euros a row that is 24,000 euros for a 0.7 improvement and 64,000 for 1.1. Here is what a 0.7
> improvement is worth to the forecast."

**Three caveats I would state with it.**

**The extrapolation assumes the new rows resemble the old ones.** If they come from a new source or
period, the failure lab applies and the curve can move the wrong way.

**Reducing capacity is free and should be tried first.** A large gap is variance, and regularisation
(05-09) attacks the same term at no data cost. If it closes the gap, the data purchase shrinks or
disappears.

**The floor limits everything.** If a flexible model on the same rows already reaches 3.0, then 3.0 is
what the extra data is buying you a slower route to, and the cheaper answer is the better model.

### E14 · Same validation RMSE, different training RMSE

**Team A** - training far below validation - **has variance.** More rows, more regularisation, or an
ensemble. Their model is finding structure that is not there, and the cheapest test is to reduce capacity
and see whether validation holds.

**Team B** - training nearly equal to validation - **has bias.** More rows are wasted. They need more
capacity: interactions, non-linear terms, or a flexible model class.

**And the sting: the two teams should swap advice.** If A says "we need more features" and B says "we
need more data", both are wrong, and the identical validation number gives neither any way to notice.

**What I would ask each for:** the learning curve. It distinguishes the two cases in one plot and neither
team needs to collect anything to draw it.

## Debugging

### E15 · A jagged learning curve

**Cause: too few folds, or too few repeats, at each sample size.** Each point is an average over `cv`
splits, and with `cv=5` on a small subset that average is itself noisy - the chapter's degree-14 curve
read 405.4 at twenty rows because one fold extrapolated wildly.

**The fixes, in order of cost:**

1. **More folds** - 10 instead of 5 - which halves the variance of each point at double the compute.
2. **Repeat the whole curve with several `random_state` values and average.** `RepeatedKFold`, or a loop.
   This is the most effective single change.
3. **Plot the spread, not just the mean.** `learning_curve` returns the per-fold scores; shading the
   range makes it obvious whether a bump is signal or noise, and stops you reading a story into it.
4. **Drop the smallest sample sizes** if the model cannot be fitted sensibly there at all.

**And a cause worth ruling out first:** if the jaggedness only appears above some size, the extra rows
may not be exchangeable with the earlier ones - which is the failure lab, not a noise problem.

### E16 · The curve looked fine and the model failed in production

**Three things a cross-validated learning curve cannot see:**

1. **Distribution shift between the data and production.** Cross-validation resamples *within* the
   dataset you have, so it measures how well the model generalises to rows like these. It says nothing
   about next quarter, a new market, or a changed upstream process. 04-04's chronological split is the
   partial answer.
2. **Leakage that is present in every fold.** If a feature contains information unavailable at prediction
   time, every fold enjoys it equally and the curve looks excellent. 04-05 covers this, and it is the
   most common cause of a large train-to-production drop.
3. **Everything that is not the metric.** Latency, a feature that is not computable in the serving path,
   a change in the input schema, silent nulls, a unit mismatch. The curve is about statistical
   performance and production failures are often not statistical.

**A fourth, if the split was random:** grouped structure. If several rows belong to one customer and the
split separates them, the model has seen that customer during training in every fold - 04-04's grouped
split - and the curve is optimistic by however much customer identity predicts the target.

## Exam and interview reasoning

### E17 · "Explain the bias-variance tradeoff."

> "The expected squared error splits into three parts. Bias is how far the model's average prediction is
> from the truth - a model too simple to represent the relationship. Variance is how much the model
> changes when you refit it on a different sample - a model flexible enough to chase the particular rows
> it got. Noise is what nobody can predict. Simple models have high bias and low variance, flexible ones
> the reverse, and the total is usually minimised somewhere in between. Practically, I read the two from
> a learning curve: the gap between training and validation error is variance, and the level they
> converge to, compared with the noise floor, is bias."

**"Modern deep networks have enormous capacity and low bias *and* low variance - does that break it?"**

> "It breaks the *picture*, not the decomposition. The identity is algebra and always holds. What is not
> universal is the U shape - very over-parameterised models can pass through a peak of high variance and
> then get *better* again as capacity grows further, which is called double descent. The usual
> explanation is that among the many fits that interpolate the training data, the optimiser prefers
> smooth ones, so the effective flexibility is far lower than the parameter count suggests.
>
> What survives is the diagnostic. I still read the gap as variance and the level as bias, and the
> actions - more data, regularisation, or a different model class - are unchanged. What does not survive
> is 'parameter count is capacity', which was always a proxy."

**What is being tested:** the first question checks you can say *variance about what*. The follow-up
checks whether you hold the decomposition as an identity or as a picture of a U - and whether you know
that the parameter-count proxy is the part that broke.

## Transfer to a different situation

### E18 · 2,000 rows, 8 euros each, budget for 3,000 more

**What I would compute before spending anything, in order - all of it on the 2,000 rows already in
hand:**

**1. The learning curve, on the model I would actually ship.** Subsets of 250, 500, 1,000, 2,000 with
repeated 5-fold. This is the whole decision and it costs an afternoon.

**2. The extrapolation to 5,000.** Fit `excess = a * n^b` to the settled part and predict. That converts
24,000 euros into an expected improvement with a number attached.

**3. The floor, two ways.** A flexible model on the same 2,000 rows, to see how much of the current error
is bias rather than noise; and if any unit was labelled twice, the disagreement between the duplicates,
which is the noise directly.

**4. The cheap alternatives to buying rows.** Regularisation, feature engineering, and a different model
class are all free and attack the same terms. If the gap closes without spending, the case for the
purchase changes.

**Then the decision follows the shape:**

- **Curves converged, well above the floor** - bias. **Do not spend.** 3,000 more rows buy nothing; the
  money should go to features or a better model.
- **Curves far apart, still falling** - variance. **Spend**, and use the extrapolation to argue for how
  many. Note that 5,000 total is only 2.5x, and a power law with exponent near -0.5 means the error falls
  by about a third of the remaining excess - modest, and worth stating plainly.
- **Curves converged at the floor** - **do not spend**, and stop working on the model entirely.

**And one thing I would buy regardless of the shape:** a few hundred rows deliberately drawn from the
*hard* part of the input space, or duplicate labels on rows you already have. Duplicates measure the
floor, and the floor is what makes every other number interpretable. That is a few hundred euros and it
tells you whether the rest of the budget has anything to buy.

## Explain it to someone non-technical

### E19 · Bias and variance without models

> Think of two darts players. The first is consistent but their aim is off - every dart lands in a tight
> cluster, an inch to the left of where they meant. The second is aiming perfectly but wobbles, so the
> darts scatter all around the target, averaging out in the right place.
>
> The first has a problem you fix by adjusting their aim; more practice throws will not help, because
> they will just repeat the same mistake. The second has a problem you fix by taking more throws and
> averaging, or by steadying their hand.
>
> Same total distance from the bullseye, and completely different remedies.

*(94 words.)* The darts analogy is standard because it is genuinely good: it carries the distinction, the
fact that the totals can be equal, and - crucially - that **the fixes are different**, which is the whole
operational point.

## Optional challenge

### E20 · Deriving the decomposition

Write $f = f(x)$ for the truth, $\hat{f}$ for the fitted model - a random quantity, because it depends on
the training sample - and $y = f + \varepsilon$ with $\mathbb{E}[\varepsilon] = 0$ and
$\text{Var}(\varepsilon) = \sigma^2$.

$$\mathbb{E}\left[(y - \hat{f})^2\right]
= \mathbb{E}\left[(f + \varepsilon - \hat{f})^2\right]
= \mathbb{E}\left[\varepsilon^2\right] + \mathbb{E}\left[(f - \hat{f})^2\right]
+ 2\,\mathbb{E}\left[\varepsilon\,(f - \hat{f})\right]$$

> **The last term is where the assumption is used.** $\varepsilon$ is the noise on the *test* point and
> $\hat{f}$ was fitted on *training* rows, so the two are independent and
> $\mathbb{E}[\varepsilon(f - \hat{f})] = \mathbb{E}[\varepsilon]\,\mathbb{E}[f - \hat{f}] = 0$.
>
> **If the test point was in the training set, this term is not zero** - the model has partly fitted that
> very noise - and the decomposition fails. That single line is why the whole of module 04 insists on
> scoring on rows the model has not seen.

Now split the second term by adding and subtracting $\bar{f} = \mathbb{E}[\hat{f}]$:

$$\mathbb{E}\left[(f - \hat{f})^2\right]
= \mathbb{E}\left[\left((f - \bar{f}) + (\bar{f} - \hat{f})\right)^2\right]
= \underbrace{(f - \bar{f})^2}_{\text{bias}^2} + \underbrace{\mathbb{E}\left[(\bar{f} - \hat{f})^2\right]}_{\text{variance}}$$

with the cross term vanishing because $(f - \bar{f})$ is a constant and
$\mathbb{E}[\bar{f} - \hat{f}] = 0$ by the definition of $\bar{f}$. Putting the pieces together:

$$\mathbb{E}\left[(y - \hat{f})^2\right] = \text{bias}^2 + \text{variance} + \sigma^2$$

**Two things the derivation shows that the statement does not.**

**It is pointwise.** Everything above is at a single $x$; the chapter's numbers are averages over an
evaluation grid, which is a choice, and a different grid gives different bias and variance.

**Nothing here is specific to linear models.** The only properties used are that squared error is a
square and that the test noise is independent of the training sample. The decomposition applies to any
model at all - and to no loss other than squared error, which is why there is no clean bias-variance
decomposition for classification accuracy.

### E21 · How does variance depend on p and on n?

In [ ]:
polynomial_variance = []
for degree in [1, 2, 4, 6]:
    for rows in [100, 200, 400, 800]:
        _, variance = terms_of(fits_for(degree, rows, 300, seed=1))
        polynomial_variance.append({"p": degree, "n": rows,
                                    "variance": variance,
                                    "variance x n": variance * rows,
                                    "variance / (noise x p / n)":
                                        variance / (NOISE_SD ** 2 * degree / rows)})
print("POLYNOMIAL features (correlated columns)")
print(pd.DataFrame(polynomial_variance).to_string(index=False,
                                                  float_format=lambda v: "%.4f" % v))

In [ ]:
def variance_with_independent_columns(p, n, datasets=400, seed=1):
    rng = np.random.default_rng(seed)
    query = rng.normal(size=(200, p))
    predictions = np.zeros((datasets, 200))
    for run in range(datasets):
        X = rng.normal(size=(n, p))
        y = X[:, 0] * 2.0 + rng.normal(0, NOISE_SD, n)
        predictions[run] = LinearRegression().fit(X, y).predict(query)
    return float(predictions.var(axis=0).mean())


independent = []
for p in [2, 4, 8]:
    for n in [100, 200, 400]:
        variance = variance_with_independent_columns(p, n)
        independent.append({"p": p, "n": n, "variance": variance,
                            "variance / (noise x p / n)":
                                variance / (NOISE_SD ** 2 * p / n)})
print("INDEPENDENT Gaussian features")
print(pd.DataFrame(independent).to_string(index=False,
                                          float_format=lambda v: "%.4f" % v))

**The `1/n` half of the claim is exact. The `p` half is only true when the columns are independent.**

**Look at `variance x n` in the first table.** Within each degree it is nearly constant - for degree 1 the
variances are 0.1329, 0.0637, 0.0327, 0.0164, halving every time `n` doubles. **Variance is inversely
proportional to the row count, and the demonstration is unambiguous.**

**Now look at the last column of the first table.** If variance were `σ² p / n` it would be constant
throughout; instead it falls steadily from about 5.9 at p=1 to about 1.4 at p=6. Read directly: at 400
rows the variance goes 0.0327, 0.0502, 0.0513, 0.0472 as p goes 1, 2, 4, 6 - it **stops growing after
p=2 entirely.**

**The second table shows why.** With independent Gaussian columns the ratio settles at about 1.2 to 1.3
for every p from 2 to 8 and every n - the `σ² p / n` law holding, up to a constant that reflects where
the query points sit.

**The difference is collinearity, and it is 05-07's conditioning problem seen from a third side.**
Standardised polynomial columns are strongly correlated with each other, so six of them do not supply six
independent directions to be estimated. The **effective** number of parameters is what enters the
variance, and it is far below the nominal count.

**Which is a genuinely useful correction to carry.** "Parameters over rows" is a serviceable rule of
thumb and it systematically *overstates* the variance of correlated features - and understates it for
features that are close to independent. When someone says a model has too many parameters for the data,
the question to ask is how many independent directions those parameters actually span.